# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [ ]:
from langchain_core.documents import Document
import fitz

# TODO: PDF 파일 경로를 입력하세요
# 예시: "../datasets/your_document.pdf"
file_path = "YOUR_PDF_FILE_PATH_HERE"

doc = fitz.open(file_path)
docs = []

# 페이지 단위로 Document 생성 (Parent Document)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": file_path.split("/")[-1],
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")

## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

In [ ]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "YOUR_COLLECTION_NAME_HERE"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

## 4. Parent Document 저장 (Docstore)

In [ ]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

## 5. Parent Document Retriever 구현

In [ ]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [ ]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "YOUR_SEARCH_QUERY_HERE"

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

# TODO: 시스템 프롬프트를 팀 문서 도메인에 맞게 수정하세요
# 예시: "당신은 의료 전문가입니다.", "당신은 법률 전문가입니다." 등
template = """
당신은 [YOUR_DOMAIN] 전문가입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.
또한, 답변에 참고한 문서의 출처와 페이지 번호를 명시하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content

print("✓ RAG 시스템 준비 완료")

## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [ ]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "YOUR_QUESTION_1_HERE",
    "YOUR_QUESTION_2_HERE",
    "YOUR_QUESTION_3_HERE"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합